# crbreak tutorial

`crbreak` looks for a spectral break in the diffuse gamma-ray emission of the Galactic plane
measured by *Fermi*-LAT. The plane is cut into windows of Galactic longitude; in each window
the foreground-subtracted spectrum is fitted with a power law (PL) and with a smooth broken
power law (SBPL), and the two are compared.

This notebook has three parts:

1. **The paper results from the shipped checkpoints.** Needs no data download and runs in seconds.
2. **The analysis step by step** on three windows: extraction, fits, nuisances, scan. Needs the
   data (`bash scripts/download_data.sh`) and takes about one minute.
3. **Full runs:** which config gives which result of the paper, and how to launch it.

**The model.** With $y = E^{2.6}\,dN/dE$ the spectrum of one window:

$$\mathrm{PL:}\quad F(E) = K\,(E/E_0)^{-\gamma}
\qquad
\mathrm{SBPL:}\quad F(E) = K\,(E/E_0)^{-\gamma_1}\left[1 + (E/E_{\rm br})^{\Delta\gamma/s}\right]^{s}$$

with $\Delta\gamma = \gamma_1 - \gamma_2 > 0$ a hardening above $E_{\rm br}$ and the smoothness
$s = 0.04$ fixed. Fits are $\chi^2$ fits, of two kinds:

- **classical:** the model is compared with the data as it is;
- **profiled:** the model is first deformed by the effective-area systematic,
  $\hat y_i(\theta, w) = \left(1 + s(E_i; w)\,S_i\right)\mu(E_i;\theta)$, where $S_i$ is the fractional
  systematic of bin $i$ and $s(E; w)$ a smooth function with a few coefficients $w$ (the *nuisances*)
  that have a prior and are profiled out at every step of the fit.

A **per-window** fit has its own parameters in every window. A **global** fit shares
$E_{\rm br}$ and $\Delta\gamma$ among all windows and keeps $K$ and $\gamma_1$ per window.
The significance of the break is the $\Delta\chi^2$ between PL and SBPL, converted with Wilks'
theorem and checked with Monte Carlo pseudo-experiments.

## 0 · Setup

The notebook works from any folder of the repository, with or without `pip install -e .`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "baseline.yaml").exists())
os.chdir(ROOT)                                   # the configs use paths relative to the repo root
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

OUT = ROOT / "results" / "tutorial"              # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)
print("repo root:", ROOT)

## 1 · Configuration

Every option of the analysis is in one YAML file, `configs/baseline.yaml`: there are no
defaults hidden in the code. Any other config starts with `extends:` and lists only the keys it
changes. `load_config` returns nested frozen dataclasses.

In [ ]:
from crbreak.config.analysis import load_config

cfg = load_config("configs/baseline.yaml")
print("sections :", ", ".join(f.name for f in cfg.__dataclass_fields__.values()))
print("windows  : |l| edges", cfg.sky.glon_edges, " |b| edges", cfg.sky.glat_edges)
print("data     :", cfg.data.kinds, "minus", cfg.data.pass8.bg_components)
print("nuisance :", cfg.nuisance.prior, "prior,", cfg.nuisance.mode, "mode,", cfg.nuisance.n_nodes, cfg.nuisance.basis, "nodes")
print("MC toys  :", cfg.mc.n_toys_window, "per window,", cfg.mc.n_toys_global_free, "global")

print("\nA variant only lists its changes:\n")
print(Path("configs/paper1/b0-2.yaml").read_text())

## 2 · The paper results, from the checkpoints

Every run writes a `checkpoint.pkl.gz` with all its fits. The checkpoints of the paper are in
`results/paper1/`:

| Folder | Run |
|---|---|
| `baseline` | 9 windows in $|l| < 85^\circ$, $|b| < 4^\circ$, 10k / 2M toys |
| `b0-2` | latitude cut $|b| < 2^\circ$ |
| `windows4`, `windows18` | 4 and 18 longitude windows |
| `ps_x2` | point-source contribution doubled |
| `unsubtracted` | no foreground subtraction |
| `iem` | the LAT interstellar emission model instead of the data |
| `piecewise` | piecewise-linear nuisance basis |
| `gaussian_prior` | Gaussian prior on the nuisances |

A checkpoint is a `dict`; `runs` holds one bundle per dataset.

In [ ]:
from crbreak.checkpoint import load_checkpoint

state = load_checkpoint("results/paper1/baseline")
run = state["runs"][0]
fitting = run.fitting                 # per-window fits
g, g_prof = run.g, run.g_prof         # global SBPL fits: classical and profiled
print("dataset :", run.kind, " windows:", len(fitting.window_fits))
print("fit names in each window:", fitting.window_fits["0000"].names)

**Per-window fits.** `fitting.window_fits[key]` is a `WindowFitResult`; its `fits[name]` are
`FitResult` objects (`params`, `chi2`, `ndof`, ...). The profiled fits are called
`PL|prof.box` and `SBPL|prof.box`. The break errors below are the $\Delta\chi^2 = 1$ profile
intervals.

In [ ]:
pl_name, sbpl_name = run.pl_prof_name, run.sbpl_prof_name

print(f"{'|l| [deg]':>10} {'Ebreak [GeV]':>22} {'dgamma':>22} {'chi2/ndof':>10} {'Z Wilks':>8} {'Z MC':>6}")
for key, wf in sorted(fitting.window_fits.items()):
    fit = wf.fits[sbpl_name]
    err = wf.profile_errors[sbpl_name]
    eb, eb_lo, eb_hi = err["Ebreak"]
    dg, dg_lo, dg_hi = err["dgamma"]
    z_wilks = wf.wilks_results[(pl_name, sbpl_name)]["Z_wilks"]
    z_mc = wf.mc_results[(pl_name, sbpl_name)].Z
    print(f"{wf.lowl:4.0f}-{wf.highl:<4.0f}  {eb:7.1f} -{eb - eb_lo:<5.1f} +{eb_hi - eb:<6.1f} "
          f"{dg:8.3f} -{dg - dg_lo:<5.3f} +{dg_hi - dg:<6.3f} {fit.chi2:6.1f}/{fit.ndof:<3d} {z_wilks:8.2f} {z_mc:6.2f}")

**Global fit.** One $E_{\rm br}$ and one $\Delta\gamma$ for the whole plane. `wilks_results` and
`mc_results` are keyed by the name of the null model.

In [ ]:
def show_global(label, gfit, null):
    print(f"{label}:  chi2/ndof = {gfit.chi2:.1f}/{gfit.ndof}")
    for name, (value, sigma) in gfit.shared_params.items():
        line = f"    {name:>7} = {value:.3f} +- {sigma:.3f} (Hessian)"
        if gfit.profile_errors and name in gfit.profile_errors:
            best, lo, hi = gfit.profile_errors[name]
            line += f"   -{best - lo:.3f} +{hi - best:.3f} (profile)"
        print(line)
    wilks, mc = gfit.wilks_results[null], gfit.mc_results[null]
    print(f"    vs {null}: delta chi2 = {wilks['delta_chi2']:.1f}, Z Wilks = {wilks['Z_wilks']:.2f}, "
          f"MC p = {mc.p_value:.2g} ({mc.n_exceed} of {mc.N} toys), Z MC = {mc.Z:.2f}\n")

show_global("classical", g, "PL")
show_global("profiled ", g_prof, pl_name)

**Figures.** `run_plots_from_checkpoint` redraws the figures of a run without refitting. Here:
the spectra with the fits, and $E_{\rm br}$, $\Delta\gamma$ against longitude next to the
$\Delta\chi^2$ contours of the global fit.

In [ ]:
from crbreak.run_plots_from_checkpoint import run_plots_from_checkpoint

saved = run_plots_from_checkpoint("results/paper1/baseline/checkpoint.pkl.gz",
                                  outdir=OUT / "baseline", only={"sed", "params_plus_contour"})
plt.close("all")
for name, path in saved.items():
    print(name, "->", Path(path).relative_to(ROOT))
    display(Image(path, width=800))

The robustness figure of the paper compares the eight variants. Its script reads the eight
checkpoints and writes `results/paper1/figures/`.

In [ ]:
subprocess.run([sys.executable, "scripts/regen_8panel_grid.py"], check=True, capture_output=True)
display(Image("results/paper1/figures/pass8_8panel_grid_params_plus_contour.png", width=800))

## 3 · The analysis step by step

From here on the data are needed: `bash scripts/download_data.sh` (see `data/README.md`).
To keep it quick this part uses the first three windows of the baseline and skips the Monte Carlo.

### 3.1 · Extraction

`SkyMesh` holds the window edges. `Data.build` averages the LAT intensity in each window,
builds every foreground component listed in `data.pass8.bg_components` and subtracts them.

In [ ]:
from dataclasses import replace
from crbreak.data_extract.main import Data, SkyMesh

cfg3 = replace(cfg, sky=replace(cfg.sky, glon_edges=(0.0, 5.0, 15.0, 25.0)))
mesh = SkyMesh(glon_lims=(cfg3.sky.glon_edges[0], cfg3.sky.glon_edges[-1]), glat_lims=cfg3.sky.glat_edges,
               glon_bins=np.array(cfg3.sky.glon_edges), glat_bins=np.array(cfg3.sky.glat_edges))

data = Data(mesh)
data.build("PASS8", cfg=cfg3, subtraction=True)      # "PASS8", "IEM" or "DRAGON"
print("windows:", {key: w.glon_bin for key, w in data.data_dict.items()})

Each window keeps what the extraction produced in `window.data`:

| Key | Content |
|---|---|
| `E_GeV` | bin energies |
| `F_total_scaled` | measured spectrum, $E^{2.6} F$ |
| `bg_components`, `F_bg_total` | each foreground component and their sum |
| `y` | `F_total_scaled - F_bg_total`: **what is fitted** |
| `sigma` | error of `y` (Poisson and effective area) |
| `Ssys_frac` | fractional effective-area systematic $S_i$, used by the nuisances |
| `n_counts` | photons per bin |

Foregrounds: isotropic (`ISO`), resolved (`PS`) and unresolved (`UPS`) point sources, the Fermi
bubbles (`BUB`), inverse Compton (`IC_cmb`, `IC_isrf`) and bremsstrahlung (`Brems_H2`, `Brems_HI`).

In [ ]:
from crbreak.plotting import plot_background_components

window = data.data_dict["0000"]
d = window.data
print(f"window |l| in {window.glon_bin} deg: {len(d['E_GeV'])} bins from {d['E_GeV'][0]:.1f} to {d['E_GeV'][-1]:.0f} GeV")
print("foreground fraction per bin:", np.round(d["F_bg_total"] / d["F_total_scaled"], 2))

fig = plot_background_components(d, sed_scaling=cfg3.data.sed_scaling)

### 3.2 · Classical fits

`Fitting` takes the extracted data. `fit_per_window` fits every model in every window and, for
each pair in `wilks_pairs`, stores the Wilks significance of the second model over the first.
`fit_global` fits all windows at once with the `shared` parameters in common.

In [ ]:
from crbreak.fitting.main import Fitting
from crbreak.fitting.models import PowerLaw, SmoothBrokenPowerLaw

pl = PowerLaw.from_config(cfg3.model)
sbpl = SmoothBrokenPowerLaw.from_config(cfg3.model)
print(pl.name, pl.param_names, "|", sbpl.name, sbpl.param_names)

fit3 = Fitting(data, solver=cfg3.solver)
fit3.fit_per_window([pl, sbpl], wilks_pairs=[("PL", "SBPL")], do_profile_errors=False)

for key, wf in fit3.window_fits.items():
    f = wf.fits["SBPL"]
    values = ", ".join(f"{n} = {v:.3g}" for n, (v, _) in f.params.items() if n != "K")
    print(f"|l| {wf.lowl:.0f}-{wf.highl:.0f}: {values}; chi2/ndof = {f.chi2:.1f}/{f.ndof}; "
          f"Z Wilks = {wf.wilks_results[('PL', 'SBPL')]['Z_wilks']:.2f}")

g3 = fit3.fit_global(sbpl, shared=("Ebreak", "dgamma"), wilks_against=["PL"], do_profile_errors=False)
print("\nglobal:", {n: f"{v:.3f} +- {e:.3f}" for n, (v, e) in g3.shared_params.items()},
      f"chi2/ndof = {g3.chi2:.1f}/{g3.ndof}, Z Wilks = {g3.wilks_results['PL']['Z_wilks']:.2f}")

### 3.3 · Profiled fits (nuisances)

A `NuisanceModel` wraps a spectral model with the deformation $s(E; w)$: `n_nodes` coefficients
on a B-spline basis in $\log E$, with a prior that is flat for $|w_j| < 1$ and Gaussian beyond
(`box_gauss`). It is passed to the same `fit_per_window` / `fit_global` calls. All its settings
come from the `nuisance` section of the config. With `mode: indep` every window has its own $w$.

In [ ]:
from crbreak.fitting.nuisance import NuisanceModel, nuisance_kwargs_from_config

nm_kwargs = nuisance_kwargs_from_config(cfg3.nuisance)
nm_pl = NuisanceModel(spectral_model=PowerLaw.from_config(cfg3.model), share_nuisance=False, **nm_kwargs)
nm_sbpl = NuisanceModel(spectral_model=SmoothBrokenPowerLaw.from_config(cfg3.model), share_nuisance=False, **nm_kwargs)

fit3.fit_per_window([nm_pl, nm_sbpl], wilks_pairs=[(nm_pl.name, nm_sbpl.name)], do_profile_errors=False)
g3_prof = fit3.fit_global(nm_sbpl, shared=("Ebreak", "dgamma"), classical_seed=g3,
                          wilks_against=[nm_pl.name], do_profile_errors=False)

for key, wf in fit3.window_fits.items():
    f = wf.fits[nm_sbpl.name]
    print(f"|l| {wf.lowl:.0f}-{wf.highl:.0f}: Ebreak = {f.params['Ebreak'][0]:.1f} GeV, dgamma = {f.params['dgamma'][0]:.3f}, "
          f"chi2 = {f.chi2:.1f}, w = {np.round(f.model.last_w, 2)}")
print("\nglobal profiled:", {n: f"{v:.3f} +- {e:.3f}" for n, (v, e) in g3_prof.shared_params.items()},
      f"chi2/ndof = {g3_prof.chi2:.1f}/{g3_prof.ndof}, Z Wilks = {g3_prof.wilks_results[nm_pl.name]['Z_wilks']:.2f}")

The nuisances absorb part of the curvature, so the significance of the break drops with respect
to the classical fit. The full pipeline also computes the $\Delta\chi^2 = 1$ profile intervals
(`do_profile_errors`, on by default; section `profile_errors` of the config).

### 3.4 · Scan of the break

`Fitting.scan` evaluates the global $\chi^2$ on a grid of $(E_{\rm br}, \Delta\gamma)$,
refitting everything else in each cell. `plot_contour_plot` draws the contours of
$\Delta\chi^2$ with respect to the global best fit.

In [ ]:
from crbreak.plotting import ContourDataset, plot_contour_plot

scan = fit3.scan(nm_sbpl, Ebreak_grid=np.linspace(5, 60, 10), dgamma_grid=np.linspace(0, 0.5, 10),
                 chi2_anchor=g3_prof.chi2, init_from_global=g3_prof, n_workers=4)
print(scan)

cr = cfg3.cr_reference
plot_contour_plot([ContourDataset.pass8(scan, g=g3_prof, label="PASS8")],
                  smooth_sigma=cfg3.plots.scan_smooth_sigma, levels=cfg3.plots.contour_levels,
                  cr_ref=(cr.ebreak_gev, cr.dgamma), cr_ebr_band=cr.ebreak_band, cr_dgamma_band=cr.dgamma_band,
                  save=OUT / "scan_3windows.png")
plt.close("all")
display(Image(OUT / "scan_3windows.png", width=450))

### 3.5 · Figures

The plotting functions take the `Fitting` object and the names of the fits to draw.

In [ ]:
from crbreak.plotting import ParamsDataset, plot_params_vs_glon, plot_sed
from crbreak.plotting.plot_sed import SedStyle

plot_sed(fit3, g=g3_prof, style=SedStyle(data_color="cornflowerblue", data_label="PASS8", ebreak_color="cornflowerblue"),
         pl_prof_name=nm_pl.name, sbpl_prof_name=nm_sbpl.name, sed_scaling=cfg3.data.sed_scaling,
         sed_band=cfg3.plots.sed_band, cr_ebr_band=cr.ebreak_band, save=OUT / "sed_3windows.png")

dataset = ParamsDataset(label="PASS8", fitting=fit3, sbpl_prof_name=nm_sbpl.name, color="cornflowerblue", marker="o")
plot_params_vs_glon([dataset], cr_ebr_band=cr.ebreak_band, cr_dgamma_band=cr.dgamma_band,
                    highb_text=cfg3.sky.glat_edges[1], save=OUT / "params_3windows.png")
plt.close("all")
display(Image(OUT / "sed_3windows.png", width=800))
display(Image(OUT / "params_3windows.png", width=450))

### 3.6 · Monte Carlo significance

Under the PL hypothesis $E_{\rm br}$ is not defined, so Wilks' theorem is not guaranteed to hold.
The pipeline therefore draws pseudo-data from the best-fit PL, refits PL and SBPL to each toy and
counts how often $\Delta\chi^2$ exceeds the observed one. It is switched on with
`do_montecarlo=True`; here with only 200 toys per window, the paper uses 10k per window and 2M
for the global fit (section `mc` of the config).

In [ ]:
fit3.fit_per_window([pl, sbpl], wilks_pairs=[("PL", "SBPL")], do_montecarlo=True, do_profile_errors=False,
                    mc_kwargs=dict(N=200, seed=cfg3.mc.seed, sample_theta=cfg3.mc.sample_theta,
                                   warm_start=cfg3.mc.warm_start, n_workers=4, batch_size=cfg3.mc.batch_size,
                                   print_every=0, store_deltas=False))
for key, wf in fit3.window_fits.items():
    mc = wf.mc_results[("PL", "SBPL")]
    print(f"|l| {wf.lowl:.0f}-{wf.highl:.0f}: delta chi2 = {mc.delta_obs:.1f}, {mc.n_exceed} of {mc.N} toys above, "
          f"p = {mc.p_value:.3g}, Z = {mc.Z:.2f}  (Wilks: {wf.wilks_results[('PL', 'SBPL')]['Z_wilks']:.2f})")

## 4 · Full runs

One command does everything above for all windows and writes figures, logs, the merged
`config.yaml` and the checkpoint to `results/runs/<date>/b<lo>-<hi>/<name>/`:

```bash
python -m crbreak.run configs/examples/pass8_nomc.yaml          # about 10 min on 20 cores
sbatch scripts/slurm/run_analysis.sbatch configs/baseline.yaml   # on a SLURM cluster
```

| Result of the paper | Config | Checkpoint |
|---|---|---|
| Baseline: spectra, $E_{\rm br}$ and $\Delta\gamma$ against $|l|$, contours, significances | `configs/baseline.yaml` | `results/paper1/baseline` |
| $|b| < 2^\circ$ | `configs/paper1/b0-2.yaml` | `results/paper1/b0-2` |
| 4 and 18 windows | `configs/paper1/windows4.yaml`, `windows18.yaml` | `results/paper1/windows4`, `windows18` |
| No foreground subtraction | `configs/paper1/unsubtracted.yaml` | `results/paper1/unsubtracted` |
| Interstellar emission model | `configs/paper1/iem.yaml` | `results/paper1/iem` |
| Piecewise-linear basis | `configs/paper1/piecewise.yaml` | `results/paper1/piecewise` |
| Gaussian prior | `configs/paper1/gaussian_prior.yaml` | `results/paper1/gaussian_prior` |
| Point sources doubled | no config key in this release | `results/paper1/ps_x2` |

The baseline with its 2M global toys is a long job (days on 20 cores); the variants use
1k / 100k toys. To make your own variant, write a YAML that extends the baseline:

```yaml
extends: ../baseline.yaml
sky:
  glat_edges: [0, 2, 4, 10]     # three |b| bands: one analysis each, plus a comparison
mc:
  do_windows: false
  do_global: false
output:
  tag: my_bands
```

Figures of a finished run can be redrawn at any time:

```bash
python -m crbreak.run_plots_from_checkpoint results/runs/<date>/b0-4/pass8/checkpoint.pkl.gz
python scripts/regen_single_checkpoint.py results/paper1/baseline    # paper-style baseline figures
python scripts/regen_8panel_grid.py                                  # robustness figure
```

## Where things are

| What | Where |
|---|---|
| Config dataclasses, `load_config` | `crbreak.config.analysis` |
| Windows and extraction: `SkyMesh`, `Data` | `crbreak.data_extract.main` |
| File readers and foreground subtraction | `crbreak.data_extract.data_loader`, `data_reduction` |
| `PowerLaw`, `SmoothBrokenPowerLaw`, `FrozenSpectralModel` | `crbreak.fitting.models` |
| `Fitting`: `fit_per_window`, `fit_global`, `scan` | `crbreak.fitting.main` |
| $\chi^2$ engines, classical and profiled | `crbreak.fitting.chi2`, `global_chi2` |
| Nuisance basis, priors, `NuisanceModel` | `crbreak.fitting.nuisance` |
| Monte Carlo toys | `crbreak.fitting.montecarlo` |
| The whole analysis in one function | `crbreak.fitting.pipeline.run` |
| Command line | `crbreak.run`, `crbreak.run_plots_from_checkpoint` |
| Figures | `crbreak.plotting` |
| Checkpoints | `crbreak.checkpoint.load_checkpoint` |